In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS ecommerce;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ecommerce.source_metadata (
    table_name STRING
);
INSERT INTO ecommerce.source_metadata (table_name) VALUES 
('users'),
('sellers'),
('buyers'),
('countries');

df = spark.table("ecommerce.source_metadata")
table_list = [row.table_name for row in df.collect()]

for table_name in table_list:
    bronze_table = f"ecommerce.{table_name}_raw"
    adls_path = f"abfss://landing@avdecommercesa.dfs.core.windows.net/{table_name}/"
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {bronze_table}
    USING PARQUET
    LOCATION '{adls_path}'
    """)
    print(f"Created table {bronze_table} from {adls_path} path")


In [0]:
# Read the parquet files and overwrite/create the table properly
df = spark.read.parquet("abfss://landing@saecommercedataprod001.dfs.core.windows.net/users/")
df.write.mode("overwrite").saveAsTable("ecommerce.users_raw")

In [0]:
df = spark.read.parquet("abfss://landing@saecommercedataprod001.dfs.core.windows.net/users/")
df.printSchema()
display(df.limit(5))